# Pipeline Logging Demo

This script demonstrates the user-facing logging added by #14. It requires a
running NDD/DataDesigner backend. Run cells individually in VS Code or PyCharm.

In [1]:
from anonymizer.logging import LoggingConfig, configure_logging  # noqa: F401

configure_logging()  # default: anonymizer INFO, DD suppressed
# configure_logging(LoggingConfig.verbose())  # anonymizer INFO + DD progress
# configure_logging(LoggingConfig.debug())    # everything DEBUG

/Users/amanoel/Code/Anonymizer/checkouts/feat-pipeline-logging/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import tempfile
from pathlib import Path

import pandas as pd

tmp_dir = tempfile.mkdtemp(prefix="logging_demo_")
csv_path = Path(tmp_dir) / "sample.csv"

pd.DataFrame(
    {
        "text": [
            "Alice Johnson works at Acme Corp in Portland, Oregon.",
            "Contact Bob Smith at bob.smith@example.com or 555-0123.",
            "Dr. Carol Lee treated patient #12345 on 2024-03-15.",
        ]
    }
).to_csv(csv_path, index=False)

print(f"Sample data saved to {csv_path}")

Sample data saved to /var/folders/66/lhpfpgl94z33ykf1njyb9y080000gp/T/logging_demo_wct5is0u/sample.csv


## Scenario 1: Full run with Redact replacement

In [3]:
from anonymizer.config.anonymizer_config import AnonymizerConfig, AnonymizerInput
from anonymizer.config.replace_strategies import Redact
from anonymizer.interface.anonymizer import Anonymizer

anonymizer = Anonymizer()
result = anonymizer.run(
    config=AnonymizerConfig(replace=Redact()),
    data=AnonymizerInput(source=str(csv_path)),
)
result.dataframe

[19:05:43] [INFO] 🔧 Anonymizer initialized with 3 model configs


[19:05:43] [INFO]   |-- 🔎 detector:  gliner-pii-detector


[19:05:43] [INFO]   |-- ✅ validator: gpt-oss-120b


[19:05:43] [INFO]   |-- 🧩 augmenter: gpt-oss-120b


[19:05:43] [INFO] 📂 Loaded 3 records from /var/folders/66/lhpfpgl94z33ykf1njyb9y080000gp/T/logging_demo_wct5is0u/sample.csv (column: 'text')


...

[19:05:54] [INFO]   |-- labels: first_name=3, last_name=3, company_name=1, city=1, state=1, email=1, phone_number=1, date=1


[19:05:54] [INFO] 🔄 Running Redact replacement


[19:05:54] [INFO]   |-- 📋 Replacement complete (0 failed) [0.0s]


[19:05:54] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


,text,text_with_spans,final_entities,text_replaced
0,"Alice Johnson works at Acme Corp in Portland, ...",<first_name>Alice</first_name> <last_name>John...,"{'entities': [{'id': 'first_name_0_5', 'value'...",[REDACTED_FIRST_NAME] [REDACTED_LAST_NAME] wor...
1,Contact Bob Smith at bob.smith@example.com or ...,Contact <first_name>Bob</first_name> <last_nam...,"{'entities': [{'id': 'first_name_8_11', 'value...",Contact [REDACTED_FIRST_NAME] [REDACTED_LAST_N...
2,Dr. Carol Lee treated patient #12345 on 2024-0...,Dr. <first_name>Carol</first_name> <last_name>...,"{'entities': [{'id': 'first_name_4_9', 'value'...",Dr. [REDACTED_FIRST_NAME] [REDACTED_LAST_NAME]...


## Scenario 2: Preview mode

In [4]:
preview = anonymizer.preview(
    config=AnonymizerConfig(replace=Redact()),
    data=AnonymizerInput(source=str(csv_path)),
    num_records=2,
)
preview.dataframe

[19:05:54] [INFO] 📂 Loaded 3 records from /var/folders/66/lhpfpgl94z33ykf1njyb9y080000gp/T/logging_demo_wct5is0u/sample.csv (column: 'text')


[19:05:54] [INFO]   |-- 👀 Preview mode: processing 2 of 3 records


[19:05:54] [INFO] 🔍 Running entity detection on 3 records


[19:06:01] [INFO]   |-- 📋 Detection complete — 2 entities found across 2 records (0 failed) [6.5s]


[19:06:01] [INFO]   |-- labels: first_name=2, last_name=2, company_name=1, city=1, state=1, email=1, phone_number=1


[19:06:01] [INFO] 🔄 Running Redact replacement


[19:06:01] [INFO]   |-- 📋 Replacement complete (0 failed) [0.0s]


[19:06:01] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


,text,text_with_spans,final_entities,text_replaced
0,"Alice Johnson works at Acme Corp in Portland, ...",<first_name>Alice</first_name> <last_name>John...,"{'entities': [{'id': 'first_name_0_5', 'value'...",[REDACTED_FIRST_NAME] [REDACTED_LAST_NAME] wor...
1,Contact Bob Smith at bob.smith@example.com or ...,Contact <first_name>Bob</first_name> <last_nam...,"{'entities': [{'id': 'first_name_8_11', 'value...",Contact [REDACTED_FIRST_NAME] [REDACTED_LAST_N...


## Scenario 3: Detection only (no replacement)

In [5]:
from anonymizer.config.anonymizer_config import Rewrite

result_detect_only = anonymizer.run(
    config=AnonymizerConfig(rewrite=Rewrite()),
    data=AnonymizerInput(source=str(csv_path)),
)
result_detect_only.dataframe

[19:06:01] [INFO] 📂 Loaded 3 records from /var/folders/66/lhpfpgl94z33ykf1njyb9y080000gp/T/logging_demo_wct5is0u/sample.csv (column: 'text')


[19:06:01] [INFO] 🔍 Running entity detection on 3 records


[19:06:16] [INFO]   |-- 📋 Detection complete — 3 entities found across 3 records (0 failed) [15.5s]


[19:06:16] [INFO]   |-- labels: first_name=3, last_name=3, company_name=1, city=1, state=1, email=1, phone_number=1, occupation=1, date=1


[19:06:16] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


,text,text_with_spans,final_entities
0,"Alice Johnson works at Acme Corp in Portland, ...",<first_name>Alice</first_name> <last_name>John...,"{'entities': [{'id': 'first_name_0_5', 'value'..."
1,Contact Bob Smith at bob.smith@example.com or ...,Contact <first_name>Bob</first_name> <last_nam...,"{'entities': [{'id': 'first_name_8_11', 'value..."
2,Dr. Carol Lee treated patient #12345 on 2024-0...,<occupation>Dr.</occupation> <first_name>Carol...,"{'entities': [{'id': 'occupation_0_3', 'value'..."


## Scenario 4: Verbose mode (surfaces DataDesigner progress)

In [6]:
configure_logging(LoggingConfig.verbose())

result_verbose = anonymizer.run(
    config=AnonymizerConfig(replace=Redact()),
    data=AnonymizerInput(source=str(csv_path)),
)
result_verbose.dataframe

[19:06:16] [INFO] 📂 Loaded 3 records from /var/folders/66/lhpfpgl94z33ykf1njyb9y080000gp/T/logging_demo_wct5is0u/sample.csv (column: 'text')


[19:06:16] [INFO] 🔍 Running entity detection on 3 records


[19:06:16] [INFO] 🎨 Creating Data Designer dataset


[19:06:16] [INFO] ✅ Validation passed


[19:06:16] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph


...

[19:06:23] [INFO]   |-- labels: first_name=3, last_name=3, company_name=1, city=1, state=1, email=1, phone_number=1, occupation=1, date=1


[19:06:23] [INFO] 🔄 Running Redact replacement


[19:06:23] [INFO]   |-- 📋 Replacement complete (0 failed) [0.0s]


[19:06:23] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


,text,text_with_spans,final_entities,text_replaced
0,"Alice Johnson works at Acme Corp in Portland, ...",<first_name>Alice</first_name> <last_name>John...,"{'entities': [{'id': 'first_name_0_5', 'value'...",[REDACTED_FIRST_NAME] [REDACTED_LAST_NAME] wor...
1,Contact Bob Smith at bob.smith@example.com or ...,Contact <first_name>Bob</first_name> <last_nam...,"{'entities': [{'id': 'first_name_8_11', 'value...",Contact [REDACTED_FIRST_NAME] [REDACTED_LAST_N...
2,Dr. Carol Lee treated patient #12345 on 2024-0...,<occupation>Dr.</occupation> <first_name>Carol...,"{'entities': [{'id': 'occupation_0_3', 'value'...",[REDACTED_OCCUPATION] [REDACTED_FIRST_NAME] [R...


## Scenario 5: Debug mode (fine-grained Anonymizer diagnostics)

In [7]:
configure_logging(LoggingConfig.debug())

result_debug = anonymizer.run(
    config=AnonymizerConfig(replace=Redact()),
    data=AnonymizerInput(source=str(csv_path)),
)
result_debug.dataframe

[19:06:23] [INFO] 📂 Loaded 3 records from /var/folders/66/lhpfpgl94z33ykf1njyb9y080000gp/T/logging_demo_wct5is0u/sample.csv (column: 'text')


[19:06:23] [INFO] 🔍 Running entity detection on 3 records


[19:06:23] [DEBUG] detection aliases: detector=gliner-pii-detector, validator=gpt-oss-120b, augmenter=gpt-oss-120b


[19:06:23] [DEBUG] NDD workflow 'entity-detection' starting with 3 records


[19:06:23] [DEBUG] NDD workflow 'entity-detection': 10 columns ['_raw_detected_entities', '_seed_entities', '_validation_candidates', '_validation_skeleton', '_validation_decisions', '_validated_entities', '_seed_entities_json', '_augmented_entities', '_merged_entities', '_detected_entities']


...

[19:06:30] [DEBUG]   record 2: 3 replacements — first_name=1, last_name=1, date=1


[19:06:30] [DEBUG] replacement stats: 12 unique entities replaced (first_name=3, last_name=3, company_name=1, city=1, state=1, email=1, phone_number=1, date=1)


[19:06:30] [INFO]   |-- 📋 Replacement complete (0 failed) [0.0s]


[19:06:30] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


,text,text_with_spans,final_entities,text_replaced
0,"Alice Johnson works at Acme Corp in Portland, ...",<first_name>Alice</first_name> <last_name>John...,"{'entities': [{'id': 'first_name_0_5', 'value'...",[REDACTED_FIRST_NAME] [REDACTED_LAST_NAME] wor...
1,Contact Bob Smith at bob.smith@example.com or ...,Contact <first_name>Bob</first_name> <last_nam...,"{'entities': [{'id': 'first_name_8_11', 'value...",Contact [REDACTED_FIRST_NAME] [REDACTED_LAST_N...
2,Dr. Carol Lee treated patient #12345 on 2024-0...,Dr. <first_name>Carol</first_name> <last_name>...,"{'entities': [{'id': 'first_name_4_9', 'value'...",Dr. [REDACTED_FIRST_NAME] [REDACTED_LAST_NAME]...


## Cleanup

This file is temporary and should be removed before merging.